# Fabric feature demo

Runs live checks against Fabric through the edge gateway (`https://inference.nscale.host/v1`) and prints the numbers that show each feature working. Each section is its own cell, so you can run them one at a time while presenting.

Needs Python 3.10+ and nothing else: the notebook imports `feature_demo.py` from this folder. Your API key is asked for with a hidden prompt and is not saved in the notebook.

In [ ]:
import getpass, os, sys
from types import SimpleNamespace

sys.path.insert(0, os.path.abspath("."))  # run this notebook from scripts/demo
import feature_demo as fd

if not os.environ.get("FABRIC_API_KEY"):
    os.environ["FABRIC_API_KEY"] = getpass.getpass("Fabric API key: ")

## Settings
Set `MODEL` to the deployment alias to test, or leave it as `None` to use the first ready model. A longer `prefix_words` makes the prefix-cache effect larger.

In [ ]:
MODEL = None   # e.g. "qwen3-coder"

args = SimpleNamespace(
    model=MODEL,
    control_url="https://api.nscale.host",
    inference_url="https://inference.nscale.host/v1",
    prefix_words=2500, prefix_max_tokens=8, repeats=5,
    stream_max_tokens=200,
    load_requests=16, concurrency=8, load_max_tokens=96,
)
demo = fd.Demo(args)

## 1. Auth
The account API key is exchanged for a short-lived inference token; the key itself never goes to the gateway.

In [ ]:
demo.demo_auth()

## 2. Models and edge routing
The models this account can call, and which stamp the edge sent the request to (`X-Fabric-Edge-Stamp`).

In [ ]:
demo.demo_models()

## 3. Prefix caching
The same long system prompt (an operations runbook of ~3,000 tokens) is sent repeatedly with different questions. A second series sends the identical prompt with a random tag at the very start, which makes every prefix unique and defeats the cache.

With prefix caching the repeated prefix is not recomputed, so its **time to first token** drops; the cached-token count comes from vLLM when it reports it. All calls carry one session key so they stay on one stamp.

In [ ]:
demo.demo_prefix()

## 4. Streaming
Time to first token, total time and decode speed for one longer answer.

In [ ]:
demo.demo_stream()

## 5. Concurrent load
Many streamed requests at once: continuous batching keeps aggregate throughput high while tail latency stays bounded.

In [ ]:
demo.demo_load()

## 6. Agent run budgets
Three calls share one run id with a budget of 2 steps. The gateway counts steps and tokens in response headers and refuses the third call.

In [ ]:
demo.demo_runs()

## 7. Auto model selection
`model: "auto"` lets the gateway pick a deployment for the prompt and say why. Needs at least one deployment with auto enabled.

In [ ]:
demo.demo_auto()

## 8. Model cascade
`model: "cascade"` tries the smallest model first and escalates only when needed. Needs two or more chat deployments.

In [ ]:
demo.demo_cascade()

## 9. Tool-call guard
A request with a tool definition: the gateway validates the model's tool calls and repairs invalid ones before they reach the client.

In [ ]:
demo.demo_guard()

## Results
Everything measured above, as one dictionary (also handy for a chart or a slide).

In [ ]:
import json
print(json.dumps(demo.report, indent=2, default=str))

### Prefix-cache chart (optional, needs matplotlib)

In [ ]:
try:
    import matplotlib.pyplot as plt
    p = demo.report["demos"].get("prefix", {})
    labels = ["first call", "repeated prefix", "unique prefix"]
    values = [(p.get(k) or 0) * 1000 for k in ("cold_ttft_s", "warm_ttft_s", "unique_ttft_s")]
    plt.bar(labels, values, color=["#999", "#222", "#c9c9c9"])
    plt.ylabel("time to first token (ms)")
    plt.title("Prefix caching")
    plt.show()
except ImportError:
    print("pip install matplotlib to draw the chart")